In [0]:

# Step 1: Define our raw-data location
RAW_PATH = "/Volumes/workspace/default/olist_raw"

# Step 2: List the files in the volume
files = dbutils.fs.ls(RAW_PATH)

# Step 3: Keep only CSV files
csv_files = {
    file.name: file
    for file in files
    if file.name.lower().endswith(".csv")
}

# Step 4: Display the results
print(f"Raw-data location: {RAW_PATH}")
print(f"CSV files found: {len(csv_files)}")

for filename, file_info in sorted(csv_files.items()):
    print(f"{filename} | {file_info.size:,} bytes")

# Step 5: Validate the expected file count
if len(csv_files) != 9:
    raise ValueError(
        f"Expected 9 CSV files, but found {len(csv_files)}."
    )

print("\nFile presence check: PASS")


Raw-data location: /Volumes/workspace/default/olist_raw
CSV files found: 9
olist_customers_dataset.csv | 9,033,957 bytes
olist_geolocation_dataset.csv | 61,273,883 bytes
olist_order_items_dataset.csv | 15,438,671 bytes
olist_order_payments_dataset.csv | 5,777,138 bytes
olist_order_reviews_dataset.csv | 14,451,670 bytes
olist_orders_dataset.csv | 17,654,914 bytes
olist_products_dataset.csv | 2,379,446 bytes
olist_sellers_dataset.csv | 174,703 bytes
product_category_name_translation.csv | 2,613 bytes

File presence check: PASS


In [0]:

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    IntegerType,
)

# Store the results of our source-data inspection
results = []

for filename, file_info in sorted(csv_files.items()):

    # Read the CSV with its header.
    # Keep all source columns as strings during initial ingestion.
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("mode", "PERMISSIVE")
        .csv(file_info.path)
    )

    # Count the records and columns
    row_count = df.count()
    column_count = len(df.columns)

    results.append((
        filename,
        row_count,
        column_count,
        file_info.size,
    ))

    print(
        f"Read {filename}: "
        f"{row_count:,} rows, {column_count} columns"
    )

# Create a summary table
summary_schema = StructType([
    StructField("file_name", StringType(), False),
    StructField("rows", LongType(), False),
    StructField("columns", IntegerType(), False),
    StructField("size_bytes", LongType(), False),
])

summary_df = spark.createDataFrame(
    results,
    schema=summary_schema,
)

display(summary_df.orderBy("file_name"))


Read olist_customers_dataset.csv: 99,441 rows, 5 columns
Read olist_geolocation_dataset.csv: 1,000,163 rows, 5 columns
Read olist_order_items_dataset.csv: 112,650 rows, 7 columns
Read olist_order_payments_dataset.csv: 103,886 rows, 5 columns
Read olist_order_reviews_dataset.csv: 104,162 rows, 7 columns
Read olist_orders_dataset.csv: 99,441 rows, 8 columns
Read olist_products_dataset.csv: 32,951 rows, 9 columns
Read olist_sellers_dataset.csv: 3,095 rows, 4 columns
Read product_category_name_translation.csv: 71 rows, 2 columns


file_name,rows,columns,size_bytes
olist_customers_dataset.csv,99441,5,9033957
olist_geolocation_dataset.csv,1000163,5,61273883
olist_order_items_dataset.csv,112650,7,15438671
olist_order_payments_dataset.csv,103886,5,5777138
olist_order_reviews_dataset.csv,104162,7,14451670
olist_orders_dataset.csv,99441,8,17654914
olist_products_dataset.csv,32951,9,2379446
olist_sellers_dataset.csv,3095,4,174703
product_category_name_translation.csv,71,2,2613


In [0]:

# Read the reviews CSV with multiline parsing enabled
reviews_path = csv_files["olist_order_reviews_dataset.csv"].path

reviews_check = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("multiLine", True)
    .csv(reviews_path)
)

print(f"Reviews row count: {reviews_check.count():,}")
print(f"Reviews column count: {len(reviews_check.columns)}")

display(reviews_check.limit(5))


Reviews row count: 99,249
Reviews column count: 7


review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,null,null,2018-01-18 00:00:00,2018-01-18 21:46:59
80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,null,null,2018-03-10 00:00:00,2018-03-11 03:05:13
228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,null,null,2018-02-17 00:00:00,2018-02-18 14:36:24
e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,null,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,null,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01 00:00:00,2018-03-02 10:26:53


In [0]:

reviews_path = csv_files["olist_order_reviews_dataset.csv"].path

parser_configs = [
    (
        "multiline_default",
        {"multiLine": "true"},
    ),
    (
        "multiline_double_quote_escape",
        {"multiLine": "true", "escape": '"'},
    ),
    (
        "multiline_back_to_delimiter",
        {
            "multiLine": "true",
            "unescapedQuoteHandling": "BACK_TO_DELIMITER",
        },
    ),
    (
        "multiline_stop_at_closing_quote",
        {
            "multiLine": "true",
            "unescapedQuoteHandling": "STOP_AT_CLOSING_QUOTE",
        },
    ),
]

for config_name, options in parser_configs:
    reader = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
    )

    for option_name, option_value in options.items():
        reader = reader.option(option_name, option_value)

    test_df = reader.csv(reviews_path)
    row_count = test_df.count()

    print(f"{config_name}: {row_count:,} rows")


multiline_default: 99,249 rows
multiline_double_quote_escape: 99,224 rows
multiline_back_to_delimiter: 99,235 rows
multiline_stop_at_closing_quote: 99,225 rows


In [0]:

# Expected row counts from our original Olist CSV files
expected_counts = {
    "olist_customers_dataset.csv": 99441,
    "olist_geolocation_dataset.csv": 1000163,
    "olist_order_items_dataset.csv": 112650,
    "olist_order_payments_dataset.csv": 103886,
    "olist_order_reviews_dataset.csv": 99224,
    "olist_orders_dataset.csv": 99441,
    "olist_products_dataset.csv": 32951,
    "olist_sellers_dataset.csv": 3095,
    "product_category_name_translation.csv": 71,
}

results = []
failures = []

for filename, file_info in sorted(csv_files.items()):

    # Configure Spark's CSV reader
    reader = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
    )

    # The reviews CSV needs special quote and multiline handling
    if filename == "olist_order_reviews_dataset.csv":
        reader = (
            reader
            .option("multiLine", True)
            .option("escape", '"')
        )

    df = reader.csv(file_info.path)

    actual_rows = df.count()
    expected_rows = expected_counts.get(filename)

    status = (
        "PASS"
        if expected_rows is not None
        and actual_rows == expected_rows
        else "FAIL"
    )

    results.append(
        (filename, expected_rows, actual_rows, status)
    )

    print(
        f"{status}: {filename} | "
        f"Expected: {expected_rows} | "
        f"Actual: {actual_rows:,}"
    )

    if status == "FAIL":
        failures.append(filename)

# Also detect missing or unexpected CSV files
missing_files = set(expected_counts) - set(csv_files)
unexpected_files = set(csv_files) - set(expected_counts)

print("\nMissing files:", sorted(missing_files))
print("Unexpected files:", sorted(unexpected_files))

if failures or missing_files or unexpected_files:
    raise ValueError(
        "Source validation failed. Investigate the files "
        "before creating Bronze tables."
    )

print("\nALL 9 SOURCE FILES VALIDATED SUCCESSFULLY")


PASS: olist_customers_dataset.csv | Expected: 99441 | Actual: 99,441
PASS: olist_geolocation_dataset.csv | Expected: 1000163 | Actual: 1,000,163
PASS: olist_order_items_dataset.csv | Expected: 112650 | Actual: 112,650


In [0]:

from pyspark.sql.functions import current_timestamp, lit

# 1. Define our raw data location
RAW_PATH = "/Volumes/workspace/default/olist_raw"

# 2. Define the expected row counts
expected_counts = {
    "olist_customers_dataset.csv": 99441,
    "olist_geolocation_dataset.csv": 1000163,
    "olist_order_items_dataset.csv": 112650,
    "olist_order_payments_dataset.csv": 103886,
    "olist_order_reviews_dataset.csv": 99224,
    "olist_orders_dataset.csv": 99441,
    "olist_products_dataset.csv": 32951,
    "olist_sellers_dataset.csv": 3095,
    "product_category_name_translation.csv": 71,
}

# 3. Map source CSV files to Bronze table names
table_names = {
    "olist_customers_dataset.csv": "customers",
    "olist_geolocation_dataset.csv": "geolocation",
    "olist_order_items_dataset.csv": "order_items",
    "olist_order_payments_dataset.csv": "order_payments",
    "olist_order_reviews_dataset.csv": "order_reviews",
    "olist_orders_dataset.csv": "orders",
    "olist_products_dataset.csv": "products",
    "olist_sellers_dataset.csv": "sellers",
    "product_category_name_translation.csv":
        "product_category_name_translation",
}

# 4. Create the Bronze schema
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")

# 5. Find the uploaded CSV files
files = dbutils.fs.ls(RAW_PATH)

csv_files = {
    file.name: file
    for file in files
    if file.name.lower().endswith(".csv")
}

if set(csv_files) != set(expected_counts):
    raise ValueError(
        "The files in the raw volume do not match "
        "the expected nine CSV files."
    )

# 6. Read, validate, and write each CSV as a Delta table
for filename, expected_rows in expected_counts.items():

    print(f"\nProcessing: {filename}")

    reader = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
    )

    # Use the parsing configuration verified earlier
    # for the reviews CSV.
    if filename == "olist_order_reviews_dataset.csv":
        reader = (
            reader
            .option("multiLine", True)
            .option("escape", '"')
        )

    df = reader.csv(csv_files[filename].path)

    # Validate the source count before writing
    actual_rows = df.count()

    if actual_rows != expected_rows:
        raise ValueError(
            f"{filename}: expected {expected_rows:,} rows, "
            f"but found {actual_rows:,}. "
            "Bronze table was not written for this file."
        )

    # Add ingestion metadata
    df = (
        df
        .withColumn("_source_file", lit(filename))
        .withColumn("_ingested_at", current_timestamp())
    )

    # Write a managed Delta table
    table_name = (
        f"workspace.bronze.{table_names[filename]}"
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

    print(
        f"SUCCESS: {table_name} | "
        f"{actual_rows:,} rows"
    )

print("\nBRONZE INGESTION COMPLETED")



Processing: olist_customers_dataset.csv
SUCCESS: workspace.bronze.customers | 99,441 rows

Processing: olist_geolocation_dataset.csv
SUCCESS: workspace.bronze.geolocation | 1,000,163 rows

Processing: olist_order_items_dataset.csv
SUCCESS: workspace.bronze.order_items | 112,650 rows

Processing: olist_order_payments_dataset.csv
SUCCESS: workspace.bronze.order_payments | 103,886 rows

Processing: olist_order_reviews_dataset.csv
SUCCESS: workspace.bronze.order_reviews | 99,224 rows

Processing: olist_orders_dataset.csv
SUCCESS: workspace.bronze.orders | 99,441 rows

Processing: olist_products_dataset.csv
SUCCESS: workspace.bronze.products | 32,951 rows

Processing: olist_sellers_dataset.csv
SUCCESS: workspace.bronze.sellers | 3,095 rows

Processing: product_category_name_translation.csv
SUCCESS: workspace.bronze.product_category_name_translation | 71 rows

BRONZE INGESTION COMPLETED


In [0]:
%sql
SHOW TABLES IN workspace.bronze;

database,tableName,isTemporary
bronze,customers,false
bronze,geolocation,false
bronze,order_items,false
bronze,order_payments,false
bronze,order_reviews,false
bronze,orders,false
bronze,product_category_name_translation,false
bronze,products,false
bronze,sellers,false
